# Life Insurance Lapse Rate Modelling

**Dataset:** Society of Actuaries — 2014 Post-Level Term Lapse and Mortality Study

Each row is an aggregated risk cell. The response is the lapse count and the exposure is policy-count exposure.

\[
L_i \mid X_i \sim \operatorname{Poisson}(E_i\lambda_i),
\qquad
\log \mathbb{E}[L_i\mid X_i]
=
\log E_i + \beta_0 + X_i^\top\beta.
\]

The development period is 2000-2001 through 2010-2011. Study year 2011-2012 is reserved for the final holdout. Model performance is assessed with an 8-fold expanding-window temporal validation.


In [ ]:
from pathlib import Path
import copy
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.metrics import mean_poisson_deviance
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("soa_post_level_term_lapse.csv")

df = pd.read_csv(DATA_PATH)
model_df = df.loc[df["EXPOSURE_CNT"] > 0].copy()

print("Raw shape:", df.shape)
print("Model shape:", model_df.shape)
print("Exposure:", model_df["EXPOSURE_CNT"].sum())
print("Lapses:", model_df["LAPSE_CNT"].sum())


## 1. Data Preparation & EDA

The descriptive summaries below reproduce the main portfolio patterns used in the project: policy duration, premium jump at duration 10, and study-year lapse experience.


In [ ]:
year_summary = (
    model_df.groupby("LAPSE_STUDY_YEAR")
    .agg(exposure=("EXPOSURE_CNT", "sum"), lapses=("LAPSE_CNT", "sum"))
)
year_summary["lapse_rate"] = year_summary["lapses"] / year_summary["exposure"]

duration_summary = (
    model_df.groupby("DURATION")
    .agg(exposure=("EXPOSURE_CNT", "sum"), lapses=("LAPSE_CNT", "sum"))
)
duration_summary["lapse_rate"] = duration_summary["lapses"] / duration_summary["exposure"]

jump_summary_d10 = (
    model_df.loc[model_df["DURATION"] == "10"]
    .groupby("PREM_JUMP_D11_D10")
    .agg(exposure=("EXPOSURE_CNT", "sum"), lapses=("LAPSE_CNT", "sum"))
)
jump_summary_d10["lapse_rate"] = (
    jump_summary_d10["lapses"] / jump_summary_d10["exposure"]
)

display(year_summary)
display(duration_summary)
display(jump_summary_d10)


## 2. Model Development

`ISSUE_YEAR_GROUP` is excluded from the final specification because new issue cohorts appear in future temporal folds and can carry material exposure.


In [ ]:
holdout_mask = model_df["LAPSE_STUDY_YEAR"] == "2011-2012"
dev_df = model_df.loc[~holdout_mask].copy()
holdout_df = model_df.loc[holdout_mask].copy()

predictors = [
    "DURATION",
    "GENDER",
    "ISSUE_AGE_GROUP",
    "FACE_AMOUNT_BAND",
    "POST_LEVEL_PREMIUM_STRUCTURE",
    "PREM_JUMP_D11_D10",
    "RISK_CLASS",
    "PREMIUM_MODE",
]

reference_categories = {
    "DURATION": "6-9",
    "GENDER": "F",
    "ISSUE_AGE_GROUP": "40-49",
    "FACE_AMOUNT_BAND": "B.  100k-249k",
    "POST_LEVEL_PREMIUM_STRUCTURE": "1. Premium Jump to ART",
    "PREM_JUMP_D11_D10": "A.  1.01 - 2.00",
    "RISK_CLASS": "Pref Resid NS",
    "PREMIUM_MODE": "1. Annual",
}

dev_years = sorted(dev_df["LAPSE_STUDY_YEAR"].unique())
folds = [
    {"train_years": dev_years[:i], "valid_year": dev_years[i]}
    for i in range(3, len(dev_years))
]
assert len(folds) == 8
assert "2011-2012" not in dev_years

glm_rows = []
oof_rows = []

encoder = OneHotEncoder(
    drop=[reference_categories[col] for col in predictors],
    handle_unknown="ignore",
    sparse_output=False,
)

for fold in folds:
    train_df = dev_df.loc[
        dev_df["LAPSE_STUDY_YEAR"].isin(fold["train_years"])
    ].copy()
    valid_df = dev_df.loc[
        dev_df["LAPSE_STUDY_YEAR"] == fold["valid_year"]
    ].copy()

    X_train = train_df[predictors]
    X_valid = valid_df[predictors]
    y_train = train_df["LAPSE_CNT"].to_numpy()
    y_valid = valid_df["LAPSE_CNT"].to_numpy()
    exposure_train = train_df["EXPOSURE_CNT"].to_numpy()
    exposure_valid = valid_df["EXPOSURE_CNT"].to_numpy()

    X_train_encoded = encoder.fit_transform(X_train)
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message="Found unknown categories", category=UserWarning
        )
        X_valid_encoded = encoder.transform(X_valid)

    X_train_glm = sm.add_constant(X_train_encoded, has_constant="add")
    X_valid_glm = sm.add_constant(X_valid_encoded, has_constant="add")

    result = sm.GLM(
        y_train,
        X_train_glm,
        family=sm.families.Poisson(),
        offset=np.log(exposure_train),
    ).fit()

    mu_glm = result.predict(
        X_valid_glm,
        offset=np.log(exposure_valid),
    )

    baseline_rate = y_train.sum() / exposure_train.sum()
    mu_baseline = exposure_valid * baseline_rate

    baseline_deviance = mean_poisson_deviance(y_valid, mu_baseline)
    glm_deviance = mean_poisson_deviance(y_valid, mu_glm)

    glm_rows.append({
        "valid_year": fold["valid_year"],
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "baseline_deviance": baseline_deviance,
        "glm_deviance": glm_deviance,
        "improvement": 1 - glm_deviance / baseline_deviance,
        "converged": result.converged,
    })

    oof_rows.append(pd.DataFrame({
        "LAPSE_STUDY_YEAR": valid_df["LAPSE_STUDY_YEAR"].to_numpy(),
        "EXPOSURE_CNT": exposure_valid,
        "LAPSE_CNT": y_valid,
        "MU_BASELINE": mu_baseline,
        "MU_GLM": mu_glm,
    }, index=valid_df.index))

chapter2_results = pd.DataFrame(glm_rows)
oof_predictions_df = pd.concat(oof_rows).sort_index()

last_validation_result = result
last_validation_encoder = copy.deepcopy(encoder)

chapter2_results


## 3. Model Evaluation & Interpretation

Performance is summarized across future validation years. Calibration is evaluated from pooled out-of-fold predictions. Rate relativities are obtained as \(\exp(eta)\) relative to the explicit reference categories.


In [ ]:
stability_summary = (
    chapter2_results[
        ["baseline_deviance", "glm_deviance", "improvement"]
    ]
    .agg(["mean", "std", "min", "max"])
    .T
)
stability_summary["CV"] = (
    stability_summary["std"] / stability_summary["mean"]
)

calibration_by_year = (
    oof_predictions_df
    .groupby("LAPSE_STUDY_YEAR")
    .agg(
        exposure=("EXPOSURE_CNT", "sum"),
        actual_lapses=("LAPSE_CNT", "sum"),
        pred_lapses_glm=("MU_GLM", "sum"),
        pred_lapses_baseline=("MU_BASELINE", "sum"),
    )
    .reset_index()
)
calibration_by_year["actual_rate"] = (
    calibration_by_year["actual_lapses"] / calibration_by_year["exposure"]
)
calibration_by_year["glm_rate"] = (
    calibration_by_year["pred_lapses_glm"] / calibration_by_year["exposure"]
)
calibration_by_year["baseline_rate"] = (
    calibration_by_year["pred_lapses_baseline"] / calibration_by_year["exposure"]
)
calibration_by_year["actual_over_predicted"] = (
    calibration_by_year["actual_lapses"]
    / calibration_by_year["pred_lapses_glm"]
)
calibration_by_year["calibration_error_pp"] = 100 * (
    calibration_by_year["actual_rate"] - calibration_by_year["glm_rate"]
)

oof_actual_rate = (
    oof_predictions_df["LAPSE_CNT"].sum()
    / oof_predictions_df["EXPOSURE_CNT"].sum()
)
oof_glm_rate = (
    oof_predictions_df["MU_GLM"].sum()
    / oof_predictions_df["EXPOSURE_CNT"].sum()
)
oof_baseline_rate = (
    oof_predictions_df["MU_BASELINE"].sum()
    / oof_predictions_df["EXPOSURE_CNT"].sum()
)
oof_actual_over_predicted = oof_actual_rate / oof_glm_rate
oof_calibration_error_pp = 100 * (oof_actual_rate - oof_glm_rate)

feature_names = last_validation_encoder.get_feature_names_out(predictors)
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": last_validation_result.params[1:],
})
coef_df["rate_relativity"] = np.exp(coef_df["coefficient"])

display(stability_summary)
display(calibration_by_year)
display(coef_df)


## 4. Final Holdout Evaluation

After development is complete, the encoder and Poisson GLM are refit once on all development years and evaluated once on 2011-2012.


In [ ]:
final_encoder = OneHotEncoder(
    drop=[reference_categories[col] for col in predictors],
    handle_unknown="ignore",
    sparse_output=False,
)

X_dev = dev_df[predictors]
y_dev = dev_df["LAPSE_CNT"].to_numpy()
exposure_dev = dev_df["EXPOSURE_CNT"].to_numpy()

X_dev_encoded = final_encoder.fit_transform(X_dev)
X_dev_glm = sm.add_constant(X_dev_encoded, has_constant="add")

final_glm = sm.GLM(
    y_dev,
    X_dev_glm,
    family=sm.families.Poisson(),
    offset=np.log(exposure_dev),
).fit()

X_holdout = holdout_df[predictors]
y_holdout = holdout_df["LAPSE_CNT"].to_numpy()
exposure_holdout = holdout_df["EXPOSURE_CNT"].to_numpy()

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Found unknown categories", category=UserWarning
    )
    X_holdout_encoded = final_encoder.transform(X_holdout)

X_holdout_glm = sm.add_constant(
    X_holdout_encoded,
    has_constant="add",
)

holdout_pred_cnt = final_glm.predict(
    X_holdout_glm,
    offset=np.log(exposure_holdout),
)

dev_portfolio_rate = y_dev.sum() / exposure_dev.sum()
baseline_pred_cnt = exposure_holdout * dev_portfolio_rate

holdout_glm_deviance = mean_poisson_deviance(
    y_holdout, holdout_pred_cnt
)
holdout_baseline_deviance = mean_poisson_deviance(
    y_holdout, baseline_pred_cnt
)
holdout_improvement = (
    1 - holdout_glm_deviance / holdout_baseline_deviance
)
holdout_actual_rate = y_holdout.sum() / exposure_holdout.sum()
holdout_predicted_rate = (
    holdout_pred_cnt.sum() / exposure_holdout.sum()
)
holdout_actual_over_predicted = (
    holdout_actual_rate / holdout_predicted_rate
)
holdout_calibration_error_pp = 100 * (
    holdout_actual_rate - holdout_predicted_rate
)

final_holdout_summary = pd.Series({
    "Baseline Poisson deviance": holdout_baseline_deviance,
    "GLM Poisson deviance": holdout_glm_deviance,
    "Improvement vs baseline": holdout_improvement,
    "Actual lapse rate": holdout_actual_rate,
    "GLM predicted rate": holdout_predicted_rate,
    "Actual / Predicted": holdout_actual_over_predicted,
    "Calibration error (p.p.)": holdout_calibration_error_pp,
})
final_holdout_summary


## 5. Final Model Assessment & Limitations

The Poisson GLM consistently outperforms the portfolio-rate baseline in out-of-time validation and on the final holdout. The main limitation is temporal calibration drift toward underprediction in later years.

Key limitations:

- aggregated risk-cell rather than individual-policy data;
- Poisson mean-variance assumption and log-linear predictor;
- fixed categorical relativities without explicit interactions;
- future-only categorical levels can occur;
- the SOA study is historical and would require fresh validation and calibration before transfer to another portfolio.


In [ ]:
final_results_summary = pd.DataFrame({
    "Metric": [
        "GLM Poisson deviance",
        "Baseline Poisson deviance",
        "Improvement vs baseline",
        "Actual / Predicted",
        "Calibration error (p.p.)",
    ],
    "Temporal validation": [
        stability_summary.loc["glm_deviance", "mean"],
        stability_summary.loc["baseline_deviance", "mean"],
        stability_summary.loc["improvement", "mean"],
        oof_actual_over_predicted,
        oof_calibration_error_pp,
    ],
    "Final holdout 2011-2012": [
        holdout_glm_deviance,
        holdout_baseline_deviance,
        holdout_improvement,
        holdout_actual_over_predicted,
        holdout_calibration_error_pp,
    ],
})
final_results_summary
